# 简化工具包

## flower

## anyio

* High level asynchronous concurrency and networking framework that works on top of either trio or asyncio
* works on top of either asyncio or trio
* https://github.com/agronholm/anyio

# asyncio

## 创建事件循环

* 事件循环是 asyncio 的核心，用于异步执行协程、处理网络请求等异步操作。
* self.loop = asyncio.new_event_loop()
* 然后可以在loop循环体中执行任务

### loop 如何一直运行

* 
```python
self.loop = asyncio.new_event_loop()

def run_async_loop(self):
    asyncio.set_event_loop(self.loop)
    self.loop.run_forever()
```

* 启动异步事件循环，这样就可以不再主线程中运行。在单独的线程中运行事件循环，确保异步操作在后台执行，而不会阻塞主线程。
threading.Thread(target=self.run_async_loop, daemon=True).start()

## call_soon_threadsafe

* self.loop.call_soon_threadsafe(self.close_connection)
* call_soon_threadsafe确保start_websocket_connection方法会在事件循环的下一次迭代中被调用，即使从其他线程调用也能保证线程安全。

## asyncio.run_coroutine_threadsafe(close(), self.loop)

* run_coroutine_threadsafe 只解决了"如何安全地将协程从其他线程提交到事件循环"这个问题，而不是解决协程本身或其访问的数据的线程安全问题。
* 协程中的线程安全： 协程本身并不保证线程安全。协程本质上是在单线程事件循环中运行，但如果多个线程共享状态（如全局变量、实例属性等），而这些状态在协程中被访问或修改，那就存在线程安全问题。

run_coroutine_threadsafe 的作用： 这个函数的主要作用不是解决协程内部的线程安全问题，而是提供一种安全的方式将协程从其他线程调度到事件循环线程中执行。它通过以下方式确保调度的安全性：

使用线程安全的队列或类似机制将协程提交到事件循环的任务队列中
保证协程最终在事件循环所在的线程中执行，而不是在调用者的线程中执行
实际的线程安全责任： 真正的线程安全需要开发者来保证：

避免在协程和其他线程之间共享可变状态
如果必须共享状态，需要使用锁或其他同步机制
协程中访问的外部状态需要是线程安全的


## handle_exception

* 因为协程是在另外一个线程中执行的，所以如果协程中出现异常，可能会导致程序异常，但不会崩溃；
  * 使得不会打印异常信息，如果不打印到log 文件将难以调试。
* 最简单的方法，try except 整个协程块